### Anomaly Detection
We run three independent ML methods on the 37 states. Any state that all three methods agree is unusual gets flagged CRITICAL. This "vote of three" approach is deliberately conservative — it reduces false alarms.

In [21]:
import pandas as pd
from pathlib import Path

OUT_DIR = Path(r"C:\Users\hp\Basic_healthcare_modelling_project\output")

features = pd.read_csv(OUT_DIR / "state_features.csv")
print(features.shape)   # (37, 18)
features.head()

(37, 18)


,state,total_q1,total_q2,total_q3,qoq_q2_pct,qoq_q3_pct,qoq_volatility,total_ytd,total_facilities,pct_l2,pct_non_functional,functionality_score,gateway_entropy,ytd_per_facility,paradox_flag,population_2023,ytd_per_million_pop,efficiency_index
0,ABIA,3.055447e+08,3.055447e+08,5.803555e+08,0.0,0.8994,1.586621e+08,1.191445e+09,190.0,0.1053,0.0,0.7118,0.751553,6.270762e+06,0,3727000,3.196793e+08,0.045478
1,ADAMAWA,4.409590e+08,4.409590e+08,8.023689e+08,0.0,0.8196,2.086601e+08,1.684287e+09,226.0,0.3186,0.0,0.8157,0.741721,7.452597e+06,0,4253000,3.960233e+08,0.051548
2,AKWA IBOM,4.157246e+08,4.157246e+08,7.918119e+08,0.0,0.9047,2.171341e+08,1.623261e+09,316.0,0.0918,0.0,0.6522,0.706286,5.136902e+06,1,5450000,2.978461e+08,0.042208
3,ANAMBRA,4.038091e+08,4.038091e+08,7.786860e+08,0.0,0.9284,2.164353e+08,1.586304e+09,328.0,0.0335,0.0,0.7232,0.741419,4.836293e+06,1,5527000,2.870100e+08,0.046986
4,BAUCHI,4.885154e+08,4.885154e+08,8.830867e+08,0.0,0.8077,2.278059e+08,1.860118e+09,212.0,0.3396,0.0,0.8052,0.776328,8.774139e+06,0,6537000,2.845522e+08,0.050365


In [22]:
# ISOLATION FOREST
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# FEATURES given to the ML models — we use RATIOS not raw totals.
# Why? Because Kano is Nigeria's most populous state and will always
# have high raw disbursements. That's not an anomaly, that's expected.
# What IS anomalous is if Kano's ₦/facility ratio is unusually high
# compared to other large states.
ML_FEATURES = [
    "ytd_per_facility",     # ₦ per PHC — funding intensity
    "ytd_per_million_pop",  # ₦ per million people — equity
    "functionality_score",  # weighted PHC health index
    "pct_non_functional",   # share of broken facilities
    "qoq_q2_pct",           # Q1→Q2 funding change %
    "qoq_q3_pct",           # Q2→Q3 funding change %
    "qoq_volatility",       # how unstable is this state's funding?
    "gateway_entropy",      # how diversified are funding channels?
    "efficiency_index",     # quality per naira composite
]
df_ml = features[["state"] + ML_FEATURES].dropna().reset_index(drop=True)
print("df_ml shape:", df_ml.shape)   # should be (37, 10)

# STANDARDISATION: scale all features to mean=0, std=1.
# Without this, a feature measured in millions (e.g. ytd_per_facility)
# would dominate features measured as fractions (e.g. pct_non_functional).
# StandardScaler makes every feature equally weighted.
scaler = StandardScaler()
X = scaler.fit_transform(df_ml[ML_FEATURES].values)

# ── ISOLATION FOREST ────
# How it works: The model builds many random decision trees.
# Normal points need many splits to isolate (they cluster together).
# Anomalous points need very few splits (they're far from the cluster).
# The fewer splits needed → the more anomalous the point.
iso_forest = IsolationForest(
    n_estimators=200,   # build 200 trees — more trees = more reliable
    contamination=0.15, # we expect about 15% of states to be anomalous (~5-6)
    random_state=42     # fix the random seed so results are reproducible
)
iso_forest.fit(X)  # train the model — it learns what "normal" looks like

# score_samples: more negative = more anomalous.
# We negate it so HIGHER = MORE ANOMALOUS (easier to understand).
df_ml["iso_score"] = -iso_forest.score_samples(X)

# predict: returns -1 for anomalies, +1 for normal points.
# We convert -1 → 1 (flagged) and +1 → 0 (clean).
df_ml["iso_flag"] = (iso_forest.predict(X) == -1).astype(int)

print(df_ml[["state", "iso_score", "iso_flag"]]
      .sort_values("iso_score", ascending=False)
      .to_string(index=False))


df_ml shape: (37, 10)
      state  iso_score  iso_flag
    KATSINA   0.603327         1
        FCT   0.584104         1
    ZAMFARA   0.582853         1
     SOKOTO   0.552217         1
       KANO   0.541600         1
      BORNO   0.534701         1
       YOBE   0.524260         0
     JIGAWA   0.502756         0
        OYO   0.497870         0
     KADUNA   0.494822         0
    BAYELSA   0.493626         0
  AKWA IBOM   0.485417         0
      LAGOS   0.476337         0
      GOMBE   0.456840         0
        IMO   0.454853         0
        EDO   0.451643         0
   NASARAWA   0.449325         0
       OSUN   0.440648         0
     TARABA   0.427998         0
      DELTA   0.425748         0
     EBONYI   0.420736         0
     BAUCHI   0.419266         0
       ABIA   0.413522         0
    ANAMBRA   0.411826         0
       OGUN   0.411662         0
    ADAMAWA   0.402799         0
       KOGI   0.402690         0
     RIVERS   0.402441         0
       ONDO   0.40230

**Key Observations**

6 states flagged — all from the North.

Katsina, FCT, Zamfara, Sokoto, Kano, and Borno are the only states the model flagged. Four of the six are in the North West and one in North East geopolitical zones. FCT is the outlier — the capital territory sitting alongside states that are among Nigeria’s most conflict-affected and highest-need regions.

The scores are tightly bunched at the top.

The gap between the top flagged state (Katsina, 0.603) and the first unflagged state (Yobe, 0.524) is small. Yobe and Jigawa are right on the boundary — they nearly made the cut. This tells you the North West/North East cluster as a whole is anomalous, not just 6 individual states.

What this likely means:
These states receive the highest raw disbursements in the country — Kano alone is Nigeria’s most populous state, and Katsina, Sokoto, Zamfara, Borno and Yobe all have large populations and significant security challenges. The model is flagging them because their funding pattern diverges from the rest of the country across multiple features simultaneously — likely driven by high ytd_per_million_pop combined with weaker functionality_score and efficiency_index. In plain terms: they’re receiving intensive funding but the facility outcomes don’t match.

The civic accountability question this raises:
For Zamfara, Sokoto and Borno especially — states with active insurgency and banditry — the gap between funding and functionality could reflect genuine security-related disruption rather than misallocation.

In [23]:
from sklearn.neighbors import LocalOutlierFactor

# LOCAL OUTLIER FACTOR: compare each state to its 5 nearest neighbours.
# A state is anomalous if its local density is much lower than its neighbours.
#
# Analogy: Imagine a city where most people are 170cm tall (tight cluster).
# Someone who's 200cm is unusual LOCALLY even if they're not unusual globally.
# LOF catches this kind of local anomaly that Isolation Forest might miss.
#
# n_neighbors=5 means "compare to the 5 most similar states".
# For 37 total states, 5 is a good neighbourhood size.
lof = LocalOutlierFactor(
    n_neighbors=5,       # how many nearby states to compare against
    contamination=0.15,  # same expectation as Isolation Forest (15% anomalous)
    metric="euclidean"   # measure distance as straight-line distance in feature space
)

# fit_predict: unlike IsolationForest, LOF does training and prediction together.
# Returns -1 = anomaly, +1 = normal
lof_labels = lof.fit_predict(X)
df_ml["lof_flag"] = (lof_labels == -1).astype(int)

# negative_outlier_factor_: more negative = more anomalous.
# We negate it so higher = more anomalous.
df_ml["lof_score"] = -lof.negative_outlier_factor_

print(df_ml[["state", "lof_score", "lof_flag"]]
      .sort_values("lof_score", ascending=False)
      .to_string(index=False))

      state  lof_score  lof_flag
    KATSINA   3.006124         1
       KANO   2.128484         1
      BORNO   2.105873         1
    ZAMFARA   1.953903         1
        FCT   1.825759         1
     SOKOTO   1.766567         1
       YOBE   1.594529         0
     KADUNA   1.416886         0
     JIGAWA   1.395170         0
      LAGOS   1.311860         0
        OYO   1.290125         0
      GOMBE   1.276498         0
  AKWA IBOM   1.184893         0
    BAYELSA   1.166741         0
      DELTA   1.115088         0
        EDO   1.114671         0
   NASARAWA   1.109297         0
       OGUN   1.101788         0
     RIVERS   1.081998         0
     BAUCHI   1.071214         0
     EBONYI   1.058719         0
     TARABA   1.043892         0
        IMO   1.028904         0
    ANAMBRA   1.017707         0
      KWARA   1.009660         0
    PLATEAU   1.006480         0
      NIGER   1.002110         0
      EKITI   1.000889         0
       KOGI   0.994104         0
    ADAMAW

**Key Observations**

The same 6 states flagged — stronger signal

Local Outlier Factor (LOF) flagged the exact same 6 states as Isolation Forest: Katsina, Kano, Borno, Zamfara, FCT, Sokoto. When two completely different algorithms — one that isolates outliers through random trees, another that compares local neighbourhood density — agree on the same 6 states, that’s a robust finding, not a coincidence.

Katsina is a clear leader.
Katsina’s LOF score of 3.006 stands well above everyone else. In LOF terms, a score above 1.5 is already significant — Katsina at 3.0 means its neighbourhood density is roughly 3 times lower than its nearest neighbours. It is genuinely isolated in the feature space, not just slightly off-centre.

The score gap is wider here.

Unlike Isolation Forest where the boundary between flagged and unflagged was tight (0.534 vs 0.524), LOF shows a clearer break — the last flagged state Sokoto scores 1.766 while Yobe just below the cut scores 1.594. The model is more confident in this separation.

The bottom of the table is telling:
States like Enugu, Cross River, Osun, Kebbi, Abia scoring below 1.0 means LOF considers them more typical than the average — they sit comfortably inside the normal cluster. These are our baseline states, the ones receiving funding proportional to their infrastructure and population.

The consolidated picture so far:
Both methods agree: the anomaly in Nigeria’s BHCPF disbursement is concentrated in the North West and North East, with FCT as the notable outlier. Let's run the Z-score method next — if it surfaces the same names, we have a defensible three-method consensus.

In [24]:
Z_THRESHOLD = 2.5  # flag if value is more than 2.5 standard deviations from mean

# Z-SCORE: for each feature, how many standard deviations is this
# state away from the average?
# Formula: z = (value - mean) / standard_deviation
# z = 0 → exactly average.  z = 2.5 → very far from average.
# z = -2.5 → very far below average.  Either extreme is a red flag.
#
# This is the simplest and most explainable method

for feat in ML_FEATURES:
    z = (df_ml[feat] - df_ml[feat].mean()) / df_ml[feat].std()
    df_ml[f"z_{feat}"] = z.round(4)  # store the z-score itself for debugging
    df_ml[f"zflag_{feat}"] = (z.abs() > Z_THRESHOLD).astype(int)  # 1 if extreme

# Count how many features are extreme for each state.
z_flag_cols = [f"zflag_{f}" for f in ML_FEATURES]
df_ml["z_feature_count"] = df_ml[z_flag_cols].sum(axis=1)

# Binary z-score flag: 1 if ANY feature is extreme, 0 if all are normal.
df_ml["z_flag"] = (df_ml["z_feature_count"] > 0).astype(int)

# ── CONSENSUS SCORE ────
# Add up how many of the 3 methods flagged each state.
# 0 = all three say CLEAN → CLEAN
# 1 = one method flagged → WATCH
# 2 = two methods agree  → CONCERN
# 3 = all three agree    → CRITICAL (highest priority for audit)
df_ml["consensus_score"] = (
    df_ml["iso_flag"] +   # 0 or 1
    df_ml["lof_flag"] +   # 0 or 1
    df_ml["z_flag"]       # 0 or 1
)                         # total: 0, 1, 2, or 3

def risk_tier(score):
    return {0: "CLEAN", 1: "WATCH", 2: "CONCERN", 3: "CRITICAL"}[score]

df_ml["risk_tier"] = df_ml["consensus_score"].apply(risk_tier)

print(df_ml[["state", "iso_flag", "lof_flag", "z_flag", "consensus_score", "risk_tier"]]
      .sort_values("consensus_score", ascending=False)
      .to_string(index=False))

      state  iso_flag  lof_flag  z_flag  consensus_score risk_tier
      BORNO         1         1       1                3  CRITICAL
       KANO         1         1       1                3  CRITICAL
    KATSINA         1         1       1                3  CRITICAL
        FCT         1         1       0                2   CONCERN
    ZAMFARA         1         1       0                2   CONCERN
     SOKOTO         1         1       0                2   CONCERN
       YOBE         0         0       1                1     WATCH
      LAGOS         0         0       1                1     WATCH
     BAUCHI         0         0       0                0     CLEAN
       ABIA         0         0       0                0     CLEAN
    ANAMBRA         0         0       0                0     CLEAN
  AKWA IBOM         0         0       0                0     CLEAN
    ADAMAWA         0         0       0                0     CLEAN
      EKITI         0         0       0                0     C

**CONSENSUS INSIGHTS**

Three states are CRITICAL — all three methods agree unanimously.
Borno, Kano, and Katsina scored 3/3. Isolation Forest, LOF, and Z-score all independently flagged them. This is the strongest possible signal the model can produce — three algorithms looking at the data from completely different mathematical angles, all pointing at the same three states

Three states are CONCERN — two methods agree.
FCT, Zamfara, and Sokoto scored 2/3. Isolation Forest and LOF both flagged them, but Z-score didn’t — meaning their individual feature values aren’t extreme enough to trip the standard deviation threshold, but their combined pattern across all 9 features is unusual enough that both clustering methods notice it. That’s a subtler kind of anomaly, worth investigating but with less urgency than the CRITICAL tier.

Two states are WATCH — Z-score only.
Yobe and Lagos are interesting. Z-score flagged them but neither Isolation Forest nor LOF did. This means one specific feature is extreme for each of them — not their overall pattern. Lagos is almost certainly flagged on ytd_per_million_pop or ytd_per_facility simply because it’s Nigeria’s largest city with unusual economics. Yobe likely has an extreme value on a security-related feature. These warrant a look at which specific feature tripped the flag.

The geographic pattern is undeniable.
Every non-CLEAN state is either in the North (Borno, Kano, Katsina, Zamfara, Sokoto, Yobe) or is FCT and Lagos — the two federal/commercial outliers. All 27 Southern and Middle Belt states are CLEAN. That’s not random. The BHCPF disbursement pattern has a clear geographic concentration that this model has now quantified.

**Why three methods?**  Each method has different blind spots. Isolation Forest catches global outliers. LOF catches local outliers (unusual only compared to nearby states). Z-Score catches single-feature extremes. If all three agree a state is anomalous, we're very confident it deserves scrutiny — the consensus approach dramatically reduces false positives.

In [26]:
# Save anomaly results for use in the Risk Model notebook
df_ml.to_csv(OUT_DIR / "anomaly_results.csv", index=False)
print("Saved anomaly_results.csv —", df_ml.shape)

Saved anomaly_results.csv — (37, 36)
